# Eliminar NANS y Dividir entre Global y Transect

In [3]:
#!/usr/bin/env python
# coding: utf-8
"""
Limpieza de datos: elimina duplicados de datetime, filas con NaNs.
Divide por transecto y guarda también archivos globales por estación.
"""

import os
import numpy as np
import pandas as pd
from pathlib import Path


INPUT_DIR = os.path.expanduser("/Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/")

OUTPUT_BY_TRANSECT = os.path.join(BASE_DIR, "imputed_by_transect")
OUTPUT_GLOBAL = os.path.join(BASE_DIR, "imputed_global")
os.makedirs(OUTPUT_BY_TRANSECT, exist_ok=True)
os.makedirs(OUTPUT_GLOBAL, exist_ok=True)

NUM_COLS = ["NO", "NO2", "NOx", "O3_for_impute", "O3",
            "Veloc.", "Direc.", "Temp.", "R.Sol.", "Dist.", "Angulo"]
MAX_SHOW_PER_COLUMN = 10
MAX_SHOW_DUPLICATE_TIMESTAMPS = 20


def load_station_data(filepath):
    df = pd.read_csv(filepath, index_col=0, parse_dates=True, low_memory=False)
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index, errors="coerce")
    return df


def detect_transect_label(df):
    if "Transecto" not in df.columns:
        return None
    tran_names = df["Transecto"].dropna().unique()
    return tran_names[0] if len(tran_names) > 0 else None


def convert_numeric_columns(df):
    df = df.copy()
    for col in NUM_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def report_nans(df, label, max_show_per_column=MAX_SHOW_PER_COLUMN):
    nan_mask = df.isna()
    total_nans = int(nan_mask.sum().sum())
    nans_by_column = nan_mask.sum()
    nans_by_column = nans_by_column[nans_by_column > 0].sort_values(ascending=False)
    print(f"\n[{label}] NaNs detectados: {total_nans}")
    if total_nans == 0:
        print(f"[{label}] No hay NaNs.")
        return total_nans
    print(f"[{label}] NaNs por columna:")
    for col, n in nans_by_column.items():
        positions = df.index[nan_mask[col]].tolist()
        preview = positions[:max_show_per_column]
        preview_str = ", ".join(str(x) for x in preview)
        more = "" if len(positions) <= max_show_per_column else f" ... (+{len(positions)-max_show_per_column} más)"
        print(f"  - {col}: {int(n)} -> [{preview_str}]{more}")
    return total_nans


def report_duplicates(df, label, max_show=MAX_SHOW_DUPLICATE_TIMESTAMPS):
    dup_mask = df.index.duplicated(keep=False)
    dup_rows = int(dup_mask.sum())
    if dup_rows == 0:
        print(f"\n[{label}] No hay duplicados de datetime.")
        return {"duplicate_rows": 0, "duplicate_timestamps": 0, "duplicate_locations": []}
    duplicated_index_values = df.index[dup_mask]
    dup_counts = pd.Series(duplicated_index_values).value_counts().sort_index()
    duplicate_timestamps = int(len(dup_counts))
    print(f"\n[{label}] Duplicados de datetime detectados: filas implicadas={dup_rows}, timestamps únicos={duplicate_timestamps}")
    # Mostrar primeros
    preview_timestamps = list(dup_counts.index[:max_show])
    duplicate_locations = []
    for ts in preview_timestamps:
        pos = np.where(df.index == ts)[0].tolist()
        duplicate_locations.append((str(ts), pos))
        pos_str = ", ".join(str(p) for p in pos[:20])
        more = "" if len(pos) <= 20 else f" ... (+{len(pos)-20} más)"
        print(f"  - {ts}: posiciones [{pos_str}]{more}")
    return {"duplicate_rows": dup_rows, "duplicate_timestamps": duplicate_timestamps, "duplicate_locations": duplicate_locations}


def clean_dataframe(df, label, station_name=None, transect_clean=None):
    df = df.copy()
    # Rellenar metadatos
    if station_name is not None:
        if "Estacion" not in df.columns:
            df["Estacion"] = station_name
        else:
            df["Estacion"] = df["Estacion"].fillna(station_name)
    if transect_clean is not None and "Transecto" in df.columns:
        df["Transecto"] = df["Transecto"].fillna(transect_clean.replace("_", " "))
    # Convertir numéricas
    df = convert_numeric_columns(df)
    original_shape = df.shape
    total_nans_before = int(df.isna().sum().sum())
    duplicate_info = report_duplicates(df, label)
    nans_before = report_nans(df, label)
    # Eliminar NaT
    nat_mask = df.index.isna()
    nat_count = int(nat_mask.sum())
    if nat_count > 0:
        print(f"\n[{label}] Filas con índice NaT: {nat_count}")
        df = df.loc[~nat_mask].copy()
    # Ordenar
    df = df.sort_index(kind="mergesort")
    # Eliminar duplicados (keep first)
    dup_mask_after_nat = df.index.duplicated(keep="first")
    duplicated_rows_removed = int(dup_mask_after_nat.sum())
    if duplicated_rows_removed > 0:
        print(f"\n[{label}] Filas duplicadas eliminadas: {duplicated_rows_removed}")
    df = df.loc[~dup_mask_after_nat].copy()
    rows_after_duplicates = df.shape[0]
    # Eliminar filas con NaN
    nan_rows_mask = df.isna().any(axis=1)
    nan_rows_removed = int(nan_rows_mask.sum())
    if nan_rows_removed > 0:
        print(f"\n[{label}] Filas con NaN eliminadas: {nan_rows_removed}")
    df = df.loc[~nan_rows_mask].copy()
    final_shape = df.shape
    print(f"\n[{label}] RESUMEN: original {original_shape} -> final {final_shape}")
    return df


def prepare_station_dataframe(df, station_name, transect_clean=None):
    df = df.copy()
    if "Estacion" not in df.columns:
        df["Estacion"] = station_name
    else:
        df["Estacion"] = df["Estacion"].fillna(station_name)
    return clean_dataframe(df, station_name, station_name, transect_clean)


def finalize_combined_dataframe(df, label):
    df = df.copy()
    nat_mask = df.index.isna()
    nat_count = int(nat_mask.sum())
    if nat_count > 0:
        print(f"\n[{label}] Índices NaT tras concatenación: {nat_count}")
        df = df.loc[~nat_mask].copy()
    df = df.sort_index(kind="mergesort")
    dup_mask = df.index.duplicated(keep="first")
    dup_count = int(dup_mask.sum())
    if dup_count > 0:
        print(f"\n[{label}] Duplicados residuales eliminados: {dup_count}")
        df = df.loc[~dup_mask].copy()
    nan_rows_mask = df.isna().any(axis=1)
    nan_count = int(nan_rows_mask.sum())
    if nan_count > 0:
        print(f"[{label}] Filas con NaN residuales eliminadas: {nan_count}")
        df = df.loc[~nan_rows_mask].copy()
    return df


def process_by_transect():
    print("\n=== Limpieza por transecto ===")
    outlier_files = list(Path(INPUT_DIR).glob("*_outliers.csv"))
    if not outlier_files:
        print("No se encontraron archivos *_outliers.csv")
        return {}, {}
    transect_dict = {}
    station_to_transect = {}
    cleaned_station_data = {}
    for filepath in outlier_files:
        station_name = filepath.stem.replace("_outliers", "")
        df_raw = load_station_data(filepath)
        if "Transecto" not in df_raw.columns:
            print(f"Advertencia: {filepath.name} no tiene Transecto. Se omite.")
            continue
        transect = detect_transect_label(df_raw)
        if transect is None:
            print(f"Advertencia: {filepath.name} no tiene valores en Transecto. Se omite.")
            continue
        transect_clean = str(transect).replace(" ", "_")
        station_to_transect[station_name] = transect_clean
        df_clean = prepare_station_dataframe(df_raw, station_name, transect_clean)
        cleaned_station_data[station_name] = df_clean
        transect_dict.setdefault(transect_clean, []).append((station_name, df_clean))
    for transect_clean, station_list in transect_dict.items():
        print(f"\n=== Procesando transecto: {transect_clean} ===")
        df_concat = pd.concat([df for _, df in station_list], axis=0, sort=False)
        df_concat = finalize_combined_dataframe(df_concat, f"TRANSECTO {transect_clean}")
        original_rows = sum(df.shape[0] for _, df in station_list)
        out_path = os.path.join(OUTPUT_BY_TRANSECT, f"{transect_clean}.csv")
        df_concat.to_csv(out_path, index=True)
        print(f"  Original: {original_rows} filas, Final: {df_concat.shape[0]} filas -> Guardado {out_path}")
    return cleaned_station_data, station_to_transect


def process_global(cleaned_station_data, station_to_transect):
    print("\n=== Limpieza global (por estación) ===")
    if not cleaned_station_data:
        return
    for station_name, df_station in cleaned_station_data.items():
        df_station = df_station.copy()
        if "Estacion" not in df_station.columns:
            df_station["Estacion"] = station_name
        if "Transecto" in df_station.columns:
            df_station["Transecto"] = df_station["Transecto"].ffill().bfill()
        df_station = finalize_combined_dataframe(df_station, f"ESTACION {station_name}")
        out_path = os.path.join(OUTPUT_GLOBAL, f"{station_name}.csv")
        df_station.to_csv(out_path, index=True)
        print(f"  Guardado: {out_path} | tamaño {df_station.shape}")


if __name__ == "__main__":
    print("Iniciando limpieza de datos...")
    cleaned, mapping = process_by_transect()
    process_global(cleaned, mapping)
    print("\nProceso completado.")

Iniciando limpieza de datos...

=== Limpieza por transecto ===

[T1_E1_Alicante] No hay duplicados de datetime.

[T1_E1_Alicante] NaNs detectados: 487
[T1_E1_Alicante] NaNs por columna:
  - O3_for_impute: 204 -> [2024-01-02 07:00:00, 2024-01-04 07:00:00, 2024-01-04 19:00:00, 2024-01-05 03:00:00, 2024-01-05 04:00:00, 2024-01-08 08:00:00, 2024-01-08 20:00:00, 2024-01-09 07:00:00, 2024-01-10 10:00:00, 2024-01-10 20:00:00] ... (+194 más)
  - Veloc.: 62 -> [2025-04-28 11:00:00, 2025-04-28 12:00:00, 2025-04-28 13:00:00, 2025-04-28 14:00:00, 2025-04-28 15:00:00, 2025-04-28 16:00:00, 2025-10-09 12:00:00, 2025-10-09 13:00:00, 2025-10-09 14:00:00, 2025-10-09 15:00:00] ... (+52 más)
  - Temp.: 62 -> [2025-04-28 11:00:00, 2025-04-28 12:00:00, 2025-04-28 13:00:00, 2025-04-28 14:00:00, 2025-04-28 15:00:00, 2025-04-28 16:00:00, 2025-10-09 12:00:00, 2025-10-09 13:00:00, 2025-10-09 14:00:00, 2025-10-09 15:00:00] ... (+52 más)
  - Direc.: 62 -> [2025-04-28 11:00:00, 2025-04-28 12:00:00, 2025-04-28 13:00

In [3]:
#!/usr/bin/env python
# coding: utf-8
"""
Limpieza de datos: elimina duplicados de datetime y filas con NaNs.
Divide por transecto y guarda también archivos globales por estación.

IMPORTANTE:
- Los duplicados de datetime se eliminan SOLO dentro de cada estación.
- Una vez concatenadas las estaciones por transecto, NO se eliminan duplicados
  de datetime, para mantener el formato original vertical.
"""

import os
import numpy as np
import pandas as pd
from pathlib import Path


INPUT_DIR = os.path.expanduser("/Volumes/copia seguridad1/enviar_benja/carpeta sin título/clean/Finales/")
BASE_DIR = INPUT_DIR

OUTPUT_BY_TRANSECT = os.path.join(BASE_DIR, "imputed_by_transect")
OUTPUT_GLOBAL = os.path.join(BASE_DIR, "imputed_global")
os.makedirs(OUTPUT_BY_TRANSECT, exist_ok=True)
os.makedirs(OUTPUT_GLOBAL, exist_ok=True)

NUM_COLS = [
    "NO", "NO2", "NOx", "O3_for_impute", "O3",
    "Veloc.", "Direc.", "Temp.", "R.Sol.", "Dist.", "Angulo"
]

MAX_SHOW_PER_COLUMN = 10
MAX_SHOW_DUPLICATE_TIMESTAMPS = 20


def load_station_data(filepath):
    df = pd.read_csv(filepath, index_col=0, parse_dates=True, low_memory=False)
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index, errors="coerce")
    return df


def detect_transect_label(df):
    if "Transecto" not in df.columns:
        return None
    tran_names = df["Transecto"].dropna().unique()
    return tran_names[0] if len(tran_names) > 0 else None


def convert_numeric_columns(df):
    df = df.copy()
    for col in NUM_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def report_nans(df, label, max_show_per_column=MAX_SHOW_PER_COLUMN):
    nan_mask = df.isna()
    total_nans = int(nan_mask.sum().sum())
    nans_by_column = nan_mask.sum()
    nans_by_column = nans_by_column[nans_by_column > 0].sort_values(ascending=False)

    print(f"\n[{label}] NaNs detectados: {total_nans}")
    if total_nans == 0:
        print(f"[{label}] No hay NaNs.")
        return total_nans

    print(f"[{label}] NaNs por columna:")
    for col, n in nans_by_column.items():
        positions = df.index[nan_mask[col]].tolist()
        preview = positions[:max_show_per_column]
        preview_str = ", ".join(str(x) for x in preview)
        more = "" if len(positions) <= max_show_per_column else f" ... (+{len(positions) - max_show_per_column} más)"
        print(f"  - {col}: {int(n)} -> [{preview_str}]{more}")
    return total_nans


def report_duplicates(df, label, max_show=MAX_SHOW_DUPLICATE_TIMESTAMPS):
    dup_mask = df.index.duplicated(keep=False)
    dup_rows = int(dup_mask.sum())

    if dup_rows == 0:
        print(f"\n[{label}] No hay duplicados de datetime.")
        return {"duplicate_rows": 0, "duplicate_timestamps": 0, "duplicate_locations": []}

    duplicated_index_values = df.index[dup_mask]
    dup_counts = pd.Series(duplicated_index_values).value_counts().sort_index()
    duplicate_timestamps = int(len(dup_counts))

    print(
        f"\n[{label}] Duplicados de datetime detectados: "
        f"filas implicadas={dup_rows}, timestamps únicos={duplicate_timestamps}"
    )

    preview_timestamps = list(dup_counts.index[:max_show])
    duplicate_locations = []
    for ts in preview_timestamps:
        pos = np.where(df.index == ts)[0].tolist()
        duplicate_locations.append((str(ts), pos))
        pos_str = ", ".join(str(p) for p in pos[:20])
        more = "" if len(pos) <= 20 else f" ... (+{len(pos) - 20} más)"
        print(f"  - {ts}: posiciones [{pos_str}]{more}")

    return {
        "duplicate_rows": dup_rows,
        "duplicate_timestamps": duplicate_timestamps,
        "duplicate_locations": duplicate_locations,
    }


def clean_dataframe(df, label, station_name=None, transect_clean=None):
    df = df.copy()

    # Rellenar metadatos
    if station_name is not None:
        if "Estacion" not in df.columns:
            df["Estacion"] = station_name
        else:
            df["Estacion"] = df["Estacion"].fillna(station_name)

    if transect_clean is not None and "Transecto" in df.columns:
        df["Transecto"] = df["Transecto"].fillna(transect_clean.replace("_", " "))

    # Convertir numéricas
    df = convert_numeric_columns(df)

    original_shape = df.shape
    _ = report_duplicates(df, label)
    _ = report_nans(df, label)

    # Eliminar NaT
    nat_mask = df.index.isna()
    nat_count = int(nat_mask.sum())
    if nat_count > 0:
        print(f"\n[{label}] Filas con índice NaT: {nat_count}")
        df = df.loc[~nat_mask].copy()

    # Ordenar
    df = df.sort_index(kind="mergesort")

    # Eliminar duplicados SOLO dentro de la estación
    dup_mask_after_nat = df.index.duplicated(keep="first")
    duplicated_rows_removed = int(dup_mask_after_nat.sum())
    if duplicated_rows_removed > 0:
        print(f"\n[{label}] Filas duplicadas eliminadas: {duplicated_rows_removed}")
        df = df.loc[~dup_mask_after_nat].copy()

    # Eliminar filas con NaN
    nan_rows_mask = df.isna().any(axis=1)
    nan_rows_removed = int(nan_rows_mask.sum())
    if nan_rows_removed > 0:
        print(f"\n[{label}] Filas con NaN eliminadas: {nan_rows_removed}")
        df = df.loc[~nan_rows_mask].copy()

    final_shape = df.shape
    print(f"\n[{label}] RESUMEN: original {original_shape} -> final {final_shape}")
    return df


def prepare_station_dataframe(df, station_name, transect_clean=None):
    df = df.copy()
    if "Estacion" not in df.columns:
        df["Estacion"] = station_name
    else:
        df["Estacion"] = df["Estacion"].fillna(station_name)
    return clean_dataframe(df, station_name, station_name, transect_clean)


def finalize_combined_dataframe(df, label):
    """
    Limpieza final del dataframe ya concatenado por transecto.
    No elimina duplicados de datetime porque en este formato vertical
    son normales al juntar estaciones distintas.
    """
    df = df.copy()

    nat_mask = df.index.isna()
    nat_count = int(nat_mask.sum())
    if nat_count > 0:
        print(f"\n[{label}] Índices NaT tras concatenación: {nat_count}")
        df = df.loc[~nat_mask].copy()

    df = df.sort_index(kind="mergesort")

    # NO eliminar duplicados aquí.
    # Ya se eliminaron dentro de cada estación antes de concatenar.
    dup_mask = df.index.duplicated(keep=False)
    dup_count = int(dup_mask.sum())
    if dup_count > 0:
        print(f"\n[{label}] Duplicados de datetime presentes tras concatenación: {dup_count}")
        print(f"[{label}] Se conservan porque pertenecen a estaciones distintas.")

    nan_rows_mask = df.isna().any(axis=1)
    nan_count = int(nan_rows_mask.sum())
    if nan_count > 0:
        print(f"[{label}] Filas con NaN residuales eliminadas: {nan_count}")
        df = df.loc[~nan_rows_mask].copy()

    return df


def process_by_transect():
    print("\n=== Limpieza por transecto ===")
    outlier_files = list(Path(INPUT_DIR).glob("*_outliers.csv"))
    if not outlier_files:
        print("No se encontraron archivos *_outliers.csv")
        return {}, {}

    transect_dict = {}
    station_to_transect = {}
    cleaned_station_data = {}

    for filepath in outlier_files:
        station_name = filepath.stem.replace("_outliers", "")
        df_raw = load_station_data(filepath)

        if "Transecto" not in df_raw.columns:
            print(f"Advertencia: {filepath.name} no tiene Transecto. Se omite.")
            continue

        transect = detect_transect_label(df_raw)
        if transect is None:
            print(f"Advertencia: {filepath.name} no tiene valores en Transecto. Se omite.")
            continue

        transect_clean = str(transect).replace(" ", "_")
        station_to_transect[station_name] = transect_clean

        df_clean = prepare_station_dataframe(df_raw, station_name, transect_clean)
        cleaned_station_data[station_name] = df_clean
        transect_dict.setdefault(transect_clean, []).append((station_name, df_clean))

    for transect_clean, station_list in transect_dict.items():
        print(f"\n=== Procesando transecto: {transect_clean} ===")

        # Concatenación vertical como antes
        df_concat = pd.concat([df for _, df in station_list], axis=0, sort=False)
        df_concat = finalize_combined_dataframe(df_concat, f"TRANSECTO {transect_clean}")

        original_rows = sum(df.shape[0] for _, df in station_list)
        out_path = os.path.join(OUTPUT_BY_TRANSECT, f"{transect_clean}.csv")
        df_concat.to_csv(out_path, index=True)

        print(f"  Original: {original_rows} filas, Final: {df_concat.shape[0]} filas -> Guardado {out_path}")

    return cleaned_station_data, station_to_transect


def process_global(cleaned_station_data, station_to_transect):
    print("\n=== Limpieza global (por estación) ===")
    if not cleaned_station_data:
        return

    for station_name, df_station in cleaned_station_data.items():
        df_station = df_station.copy()

        if "Estacion" not in df_station.columns:
            df_station["Estacion"] = station_name

        if "Transecto" in df_station.columns:
            df_station["Transecto"] = df_station["Transecto"].ffill().bfill()

        df_station = finalize_combined_dataframe(df_station, f"ESTACION {station_name}")

        out_path = os.path.join(OUTPUT_GLOBAL, f"{station_name}.csv")
        df_station.to_csv(out_path, index=True)
        print(f"  Guardado: {out_path} | tamaño {df_station.shape}")


if __name__ == "__main__":
    print("Iniciando limpieza de datos...")
    cleaned, mapping = process_by_transect()
    process_global(cleaned, mapping)
    print("\nProceso completado.")

Iniciando limpieza de datos...

=== Limpieza por transecto ===

[T1_E1_Alicante] No hay duplicados de datetime.

[T1_E1_Alicante] NaNs detectados: 0
[T1_E1_Alicante] No hay NaNs.

[T1_E1_Alicante] RESUMEN: original (138250, 12) -> final (138250, 12)

[T1_E2_Elda] No hay duplicados de datetime.

[T1_E2_Elda] NaNs detectados: 0
[T1_E2_Elda] No hay NaNs.

[T1_E2_Elda] RESUMEN: original (56160, 12) -> final (56160, 12)

[T2_E1_Elche] No hay duplicados de datetime.

[T2_E1_Elche] NaNs detectados: 0
[T2_E1_Elche] No hay NaNs.

[T2_E1_Elche] RESUMEN: original (127239, 12) -> final (127239, 12)

[T2_E2_Elda] No hay duplicados de datetime.

[T2_E2_Elda] NaNs detectados: 0
[T2_E2_Elda] No hay NaNs.

[T2_E2_Elda] RESUMEN: original (56160, 12) -> final (56160, 12)

[T3_E1_Valencia] No hay duplicados de datetime.

[T3_E1_Valencia] NaNs detectados: 0
[T3_E1_Valencia] No hay NaNs.

[T3_E1_Valencia] RESUMEN: original (128949, 12) -> final (128949, 12)

[T3_E2_Buñol] No hay duplicados de datetime.

[T